<a href="https://colab.research.google.com/github/arshpreetw11/Vision-Transformer/blob/main/replicating_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##CREATING A VISION TRANSFORMER!!

In [ ]:
try:
  import torch
  import torchvision
  assert int(torch.__version__.split(".")[1]) >= 12, "torch version should be 1.12+"
  assert int(torchvision.__version__.split(".")[1]) >= 13, "torchvision version should be 0.13+"
  print(f"torch version: {torch.__version__}")
  print(f"torchvision version: {torchvision.__version__}")
except:
  print("[INFO] torch/torchvision versions not as required ,installing nightly versions.")
  !pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
  import torch
  import torchvision
  print(f"torch version: {torch.__version__}")
  print(f"torchvision version: {torchvision.__version__}")

In [ ]:
import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn
from torchvision import transforms
try:
  from torchinfo import summary
except:
  print("[INFO] Couldn't find torchinfo,installing it...")
  !pip install -q torchinfo
  from torchinfo import summary

try:
  from going_modular.going_modular import data_steup,engine
  from helper_functions import download_data,set_seeds,plot_loss_curves

except:
  print("[INFO] Couldn't find going_modular or helper_functions scripts... downloading them from GitHub.")
  !git clone https://github.com/mrdbourke/pytorch-deep-learning
  !mv pytorch-deep-learning/going_modular .
  !mv pytorch-deep-learning/helper_functions.py . # get the helper_functions.py script
  !rm -rf pytorch-deep-learning
  from going_modular.going_modular import data_setup, engine
  from helper_functions import download_data, set_seeds, plot_loss_curves

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
image_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")
image_path

In [ ]:
train_dir=image_path/"train"
test_dir=image_path/"test"

In [ ]:
IMG_SIZE=224
manual_transform=transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor()
])
print(f"manual transform:{manual_transform}")

In [ ]:
BATCH_SIZE=32
train_dataloader,test_dataloader,class_names=data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=manual_transform,
    batch_size=BATCH_SIZE
)
train_dataloader,test_dataloader,class_names

In [ ]:
image_batch,label_batch=next(iter(train_dataloader))
image,label=image_batch[0],label_batch[0]
image.shape,label

In [ ]:
plt.imshow(image.permute(1,2,0))
plt.title(class_names[label])
plt.axis(False)

In [ ]:
height=224
width=224
patch_size=16
color_channels=3

num_of_patchs=int((height*width)/(patch_size**2))
print(f"Number of patches with height and width {height} ,{width} and patch size {patch_size} is {num_of_patchs}")


In [ ]:
embed_layer_input_shape=(height,width,color_channels)
embed_layer_output_shape=(num_of_patchs,patch_size**2*color_channels)

print(f"Input shape of embedding layer is {embed_layer_input_shape}")
print(f"Output shape of embedding layer is {embed_layer_output_shape}")

In [ ]:
plt.imshow(image.permute(1,2,0))
plt.title(class_names[label])
plt.axis(False)

In [ ]:
image_permuted=image.permute(1,2,0)
IMG_SIZE=224
patch_size=16
num_patches=IMG_SIZE/patch_size
plt.figure(figsize=(patch_size,patch_size))
assert IMG_SIZE % patch_size==0, "Image size must be divisible by patch size"
print(f"Number of patches per row: {num_patches}\nPatch size: {patch_size} pixels x {patch_size} pixels")

fig, axs = plt.subplots(nrows=1,
                        ncols=IMG_SIZE // patch_size, # one column for each patch
                        figsize=(num_patches, num_patches),
                        sharex=True,
                        sharey=True)
for i, patch in enumerate(range(0, IMG_SIZE, patch_size)):
    axs[i].imshow(image_permuted[:patch_size, patch:patch+patch_size, :]); # keep height index constant, alter the width index
    axs[i].set_xlabel(i+1) # set the label
    axs[i].set_xticks([])
    axs[i].set_yticks([])

In [ ]:
img_size=224
patch_size=16
num_of_patchs=img_size/patch_size
assert img_size % patch_size == 0, "Image size must be divisible by patch size"
print(f"Number of patches per row: {num_patches}\
        \nNumber of patches per column: {num_patches}\
        \nTotal patches: {num_patches*num_patches}\
        \nPatch size: {patch_size} pixels x {patch_size} pixels")
fig, axs = plt.subplots(nrows=img_size // patch_size,
                        ncols=img_size // patch_size,
                        figsize=(num_patches, num_patches),
                        sharex=True,
                        sharey=True)
for i,patch_height in enumerate(range(0,img_size,patch_size)):
    for j,patch_width in enumerate(range(0,img_size,patch_size)):

      axs[i, j].imshow(image_permuted[patch_height:patch_height+patch_size, # iterate through height
                                        patch_width:patch_width+patch_size, # iterate through width
                                        :])
      axs[i, j].set_ylabel(i+1,
                             rotation="horizontal",
                             horizontalalignment="right",
                             verticalalignment="center")
      axs[i, j].set_xlabel(j+1)
      axs[i, j].set_xticks([])
      axs[i, j].set_yticks([])
      axs[i, j].label_outer()
fig.suptitle(f"{class_names[label]} -> Patchified", fontsize=16)
plt.show()


In [ ]:
from torch import nn
patch_size=16
conv2d=nn.Conv2d(in_channels=3,
                 out_channels=768,
                 kernel_size=patch_size,
                 stride=patch_size,
                 padding=0)

In [ ]:
plt.imshow(image.permute(1,2,0))
plt.title(class_names[label])
plt.axis(False)

In [ ]:
image_out_of_conv2d=conv2d(image.unsqueeze(dim=0))
image_out_of_conv2d.shape

In [ ]:
import random
random_indexes=random.sample(range(0,758),5)
fig, axs = plt.subplots(nrows=1,
                        ncols=5,
                        figsize=(12, 12))
for i,idx in enumerate(random_indexes):
  image_conv_feature_map = image_out_of_conv2d[:, idx, :, :] # index on the output tensor of the convolutional layer
  axs[i].imshow(image_conv_feature_map.squeeze().detach().numpy())
  axs[i].set(xticklabels=[], yticklabels=[], xticks=[], yticks=[]);

In [ ]:
single_feature_map=image_out_of_conv2d[:,0,:,:]
single_feature_map,single_feature_map.requires_grad

In [ ]:
print(f"Current tensor shape {single_feature_map.shape}")

In [ ]:
flatten=nn.Flatten(start_dim=2,
                   end_dim=3)

In [ ]:
plt.imshow(image.permute(1, 2, 0)) # adjust for matplotlib
plt.title(class_names[label])
plt.axis(False);
print(f"Original image shape: {image.shape}")

image_out_of_conv = conv2d(image.unsqueeze(0)) # add batch dimension to avoid shape errors
print(f"Image feature map shape: {image_out_of_conv.shape}")

image_out_of_flatten = flatten(image_out_of_conv)
print(f"Flattened image feature map shape: {image_out_of_flatten.shape}")

In [ ]:
image_out_of_conv_flattened_reshaped=image_out_of_flatten.permute(0,2,1)
print(f"Patch embedding sequence shape {image_out_of_conv_flattened_reshaped.shape}")

In [ ]:
single_flatten_feature_map=image_out_of_conv_flattened_reshaped[:,:,0]
plt.figure(figsize=(22, 22))
plt.imshow(single_flatten_feature_map.detach().numpy())
plt.title(f"Flattened feature map shape: {single_flatten_feature_map.shape}")
plt.axis(False);

In [ ]:
class PatchEmbedding(nn.Module):
  def __init__(self,
               in_channels:int=3,
               patch_size:int=16,
               embedding_dim:int=768
               ):
    super().__init__()

    self.parcher=nn.Conv2d(in_channels=in_channels,
                                 out_channels=embedding_dim,
                                 kernel_size=patch_size,
                                 stride=patch_size,
                                 padding=0
    )
    self.flatten=nn.Flatten(start_dim=2,
                         end_dim=3)
  def forward(self,x):
    image_resolution = x.shape[-1]
    assert image_resolution % patch_size == 0, f"Input image size must be divisible by patch size, image shape: {image_resolution}, patch size: {patch_size}"

    x_patched=self.parcher(x)
    x_flattened = self.flatten(x_patched)

    return x_flattened.permute(0, 2, 1)

In [ ]:
set_seeds()

patchify=PatchEmbedding(in_channels=3,
                          patch_size=16,
                          embedding_dim=768)
print(f"Input image shape: {image.unsqueeze(0).shape}")
patch_embedded_image = patchify(image.unsqueeze(0))
print(f"Output patch embedding shape: {patch_embedded_image.shape}")

In [ ]:
random_input_image = (1, 3, 224, 224)
random_input_image_error = (1, 3, 250, 250)

In [ ]:
summary(PatchEmbedding(),
        input_size=random_input_image,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

In [ ]:
print(patch_embedded_image)
print(f"Patch embedding shape: {patch_embedded_image.shape}")

In [ ]:
batch_size=patch_embedded_image.shape[0]
embedding_dim=patch_embedded_image.shape[-1]

class_token=nn.Parameter(torch.ones(batch_size,1,embedding_dim),
                         requires_grad=True)
print(class_token[:,:,:10])
print(f"Class token shape: {class_token.shape}")

In [ ]:
patch_image_with_class_token=torch.cat((class_token,patch_embedded_image),dim=1)
print(patch_image_with_class_token[:,:,:10])
print(f"Patch embedding shape: {patch_image_with_class_token.shape}")
#

In [ ]:
number_of_patches=int((height*width)/patch_size**2)
embedding_dim=patch_image_with_class_token.shape[2]

position_embedding=nn.Parameter(torch.ones(1,number_of_patches+1,embedding_dim),
                                requires_grad=True)
print(position_embedding[:,:10,:10])
print(f"Position embedding shape: {position_embedding.shape}")

In [ ]:
patch_position_emb=patch_image_with_class_token+position_embedding
print(patch_position_emb[:,:,:10])
print(f"patch position embedding shape: {patch_position_emb.shape}")

In [ ]:
set_seeds()

patch_size=16

print(f"Image tensor shape {image.shape}")
height,width=image.shape[1],image.shape[2]

x=image.unsqueeze(0)
print(f"Image tensor with batch dimension shape: {x.shape}")

patchify=PatchEmbedding(in_channels=3,
                        patch_size=patch_size,
                        embedding_dim=768)
print(f"Output patch embedding shape: {patchify(x).shape}")

batch_size=patchify(x).shape[0]
embedding_dim=patchify(x).shape[-1]

class_token=nn.Parameter(torch.ones(batch_size,1,embedding_dim),
                         requires_grad=True)
print(f"class token shape: {class_token.shape}")

patch_image_with_class_token=torch.cat((class_token,patchify(x)),dim=1)
print(f"patch image with class token shape: {patch_image_with_class_token.shape}")

number_of_patches=int((height*width)/patch_size**2)
embedding_dim=patch_image_with_class_token.shape[2]
position_embedding=nn.Parameter(torch.ones(1,number_of_patches+1,embedding_dim),
                                requires_grad=True)
print(f"position embedding shape: {position_embedding.shape}")

patch_position_emb=patch_image_with_class_token+position_embedding
print(f"patch position embedding shape: {patch_position_emb.shape}")


In [ ]:
class MultiHeadSelfAttentionBoard(torch.nn.Module):
  def __init__(self,
               embedding_dim:int=768,
               num_heads:int=12,
               dropout:float=0):
    super().__init__()

    self.layer_norm=nn.LayerNorm(normalized_shape=embedding_dim)
    self.multi_head_attention=nn.MultiheadAttention(embed_dim=embedding_dim,
                                                     num_heads=num_heads,
                                                     dropout=dropout,
                                                    batch_first=True)
  def forward(self,x):
    x=self.layer_norm(x)
    atten,_=self.multi_head_attention(query=x,
                                      key=x,
                                      value=x,need_weights=False)
    return atten

In [ ]:
multi_head=MultiHeadSelfAttentionBoard(embedding_dim=768,
                                       num_heads=12)
patched_image_through_msa=multi_head(patch_position_emb)
print(f"patched image through msa shape: {patched_image_through_msa.shape}")

In [ ]:
class MLPBlock(torch.nn.Module):
    def __init__(self,embedding_dim:int=768,
                 mlp_size:int=3072,
                 dropout:float=0.1):
      super().__init__()

      self.layer_norm=nn.LayerNorm(normalized_shape=embedding_dim)

      self.mlp=nn.Sequential(
          nn.Linear(in_features=embedding_dim,
                    out_features=mlp_size),
          nn.GELU(),
          nn.Dropout(p=dropout),
          nn.Linear(in_features=mlp_size,
                    out_features=embedding_dim),
          nn.Dropout(p=dropout)
      )
    def forward(self,x):
      x=self.layer_norm(x)
      x=self.mlp(x)
      return x



In [ ]:
mlp_block=MLPBlock(embedding_dim=768,mlp_size=3072,dropout=0.1)
patched_image_through_mlp=mlp_block(patched_image_through_msa)
print(f"patched image through mlp shape: {patched_image_through_mlp.shape}")

In [ ]:
class TransformerEncoderBlock(torch.nn.Module):
  def __init__(self,
               embedding_dim:int=768,
               num_heads:int=12,
               mlp_size:int=307,
               mlp_dropout:float=0.1,
               attn_dropout:float=0):
    super().__init__()

    self.msa=MultiHeadSelfAttentionBoard(embedding_dim=embedding_dim,
                                         num_heads=num_heads,
                                         dropout=attn_dropout)
    self.mlp=MLPBlock(embedding_dim=embedding_dim,
                      mlp_size=mlp_size,
                      dropout=mlp_dropout)
  def forward(self,x):
    x=x+self.msa(x)
    x=x+self.mlp(x)
    return x

In [ ]:
transformer_encoder_block=TransformerEncoderBlock()


In [ ]:
torch_transformer_encoder_block=torch.nn.TransformerEncoderLayer(d_model=768,
                                                                 nhead=12,
                                                                 dim_feedforward=3072,
                                                                 dropout=0.1,
                                                                 activation="gelu",
                                                                 batch_first=True,
                                                                 norm_first=True)
torch_transformer_encoder_block

In [ ]:
class ViT(torch.nn.Module):
  def __init__(self,
               img_size:int=224,
               in_channels:int=3,
               patch_size:int=16,
               num_transformer_layers:int=12,
               embedding_dim:int=768,
               mlp_size:int=3072,
               num_heads:int=1,
               attn_dropout:float=0,
               mlp_dropout:float=0.1,
               emb_dropout:float=0.1,
               num_classes:int=1000):
    super().__init__()

    assert img_size % patch_size == 0, "Image size must be divisible by patch size"

    self.num_patches=(img_size*img_size)//patch_size**2
    self.class_embedding=torch.nn.Parameter(torch.randn(1,1,embedding_dim),
                                            requires_grad=True)
    self.position_embedding=torch.nn.Parameter(torch.randn(1,self.num_patches+1,embedding_dim),
                                            requires_grad=True)
    self.embedding_dropout=torch.nn.Dropout(p=emb_dropout)
    self.patch_embedding=PatchEmbedding(in_channels=in_channels,
                                        patch_size=patch_size,
                                        embedding_dim=embedding_dim)
    self.transformer_encoder=nn.Sequential(*[TransformerEncoderBlock(embedding_dim=embedding_dim,
                                                                num_heads=num_heads,
                                                                mlp_size=mlp_size,
                                                                     mlp_dropout=mlp_dropout) for _ in range(num_transformer_layers)])
    self.classifier=torch.nn.Sequential(
        torch.nn.LayerNorm(normalized_shape=embedding_dim),
        torch.nn.Linear(in_features=embedding_dim,
                        out_features=num_classes)
    )
  def forward(self,x):
    batch_size=x.shape[0]
    class_token=self.class_embedding.expand(batch_size,-1,-1)
    x=self.patch_embedding(x)
    x=torch.cat((class_token,x),dim=1)
    x=x+self.position_embedding
    x=self.embedding_dropout(x)
    x=self.transformer_encoder(x)
    x=self.classifier(x[:,0])
    return x

In [ ]:
batch_size=32

class_token_emb_single=nn.Parameter(data=torch.randn(1,1,768))
class_token_emb_batch=class_token_emb_single.expand(batch_size,-1,-1)
class_token_emb_batch.shape

In [ ]:
set_seeds()

random_image_tensor=torch.randn(1,3,224,224)
vit=ViT(num_classes=len(class_names))
vit(random_image_tensor)

In [ ]:
summary(vit,
        input_size=(1,3,224,224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

In [ ]:
from going_modular.going_modular import engine

optimizer=torch.optim.Adam(params=vit.parameters(),
                           lr=3e-3,
                           betas=(0.9,0.999),
                           weight_decay=0.3)
loss_fn=torch.nn.CrossEntropyLoss()
set_seeds()
results=engine.train(model=vit,
                     train_dataloader=train_dataloader,
                     test_dataloader=test_dataloader,
                     optimizer=optimizer,
                     loss_fn=loss_fn,
                     epochs=10,
                     device=device)


In [ ]:
from helper_functions import plot_loss_curves
plot_loss_curves(results)

In [ ]:
import torch
import torchvision
print(f'Torch version: {torch.__version__}')
print(f'Torchvision version: {torchvision.__version__}')


In [ ]:
pretrained_vit_weights=torchvision.models.ViT_B_16_Weights.DEFAULT

pretrained_vit=torchvision.models.vit_b_16(weights=pretrained_vit_weights)

for param in pretrained_vit.parameters():
  param.requires_grad=False

set_seeds()
pretrained_vit.heads=torch.nn.Linear(in_features=768,
                                     out_features=len(class_names)).to(device)



In [ ]:
summary(pretrained_vit,
        input_size=(1,3,224,224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

In [ ]:
from helper_functions import download_data
image_path=download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")
image_path

In [ ]:
train_dir=image_path/"train"
test_dir=image_path/"test"

In [ ]:
pretrained_vit_transforms=pretrained_vit_weights.transforms()

In [ ]:
train_dataloader,test_dataloader,class_names=data_setup.create_dataloaders(train_dir=train_dir,
                                                                            test_dir=test_dir,
                                                                            transform=pretrained_vit_transforms,
                                                                            batch_size=32)



In [ ]:
from going_modular.going_modular import engine

optimizer=torch.optim.Adam(params=pretrained_vit.parameters(),
                           lr=1e-3
                           )
loss_fn=torch.nn.CrossEntropyLoss()
set_seeds()
results_pretrained_vit=engine.train(model=pretrained_vit,
                     train_dataloader=train_dataloader,
                     test_dataloader=test_dataloader,
                                    optimizer=optimizer,
                                    loss_fn=loss_fn,
                                    epochs=10,
                                    device=device)


In [ ]:
plot_loss_curves(results_pretrained_vit)

In [ ]:
from going_modular.going_modular import utils

utils.save_model(model=pretrained_vit,
                 target_dir="models",
                 model_name="pretrained_vit_feature_extractor_pizza_steak_sushi.pth")

In [ ]:
from pathlib import Path

pretrained_vit_model_size=Path("models/pretrained_vit_feature_extractor_pizza_steak_sushi.pth").stat().st_size // (1024*1024) # division converts bytes to megabytes (roughly)
print(f"Pretrained ViT feature extractor model size: {pretrained_vit_model_size} MB")

In [ ]:
import requests
from pathlib import Path

from going_modular.going_modular.predictions import pred_and_plot_image

custom_image_path = image_path / "04-pizza-dad.jpeg"

# Ensure the file is fresh by deleting it if it exists
if custom_image_path.is_file():
  print(f"Deleting existing file: {custom_image_path}")
  custom_image_path.unlink() # Delete the file

if not custom_image_path.is_file():
  with open(custom_image_path, "wb") as f:
    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/04-pizza-dad.jpeg")
    print("Downloading image...")
    f.write(request.content)
else:
  print(f"{custom_image_path} already exists (this should not happen after deletion)")

pred_and_plot_image(model=pretrained_vit,
                    image_path=custom_image_path,
                    class_names=class_names,
                    transform=pretrained_vit_transforms)